Connected to reactive-agent (3.11.x) (Python 3.11.-1)

 # Calculator tool

 **One job:** evaluate math expressions safely — without ever calling `eval()` on untrusted input.

 In an agent context the LLM constructs the expression string.
 A bare `eval()` would execute anything the LLM generates, including
 `__import__('os').system('rm -rf /')`. AST parsing restricts execution
 to a whitelist of operators — nothing else can run.


In [ ]:
import ast
import operator as op
from typing import Any
from langchain_core.tools import tool
from app.core.logging import get_logger

log = get_logger(__name__)

 ## `_ALLOWED_OPERATORS`

 Whitelist of permitted AST node types mapped to their Python operator functions.
 Any operator not in this dict is rejected at the AST level — before any computation.

In [ ]:
_ALLOWED_OPERATORS = {
    ast.Add:      op.add,
    ast.Sub:      op.sub,
    ast.Mult:     op.mul,
    ast.Div:      op.truediv,
    ast.Pow:      op.pow,
    ast.USub:     op.neg,
    ast.Mod:      op.mod,
    ast.FloorDiv: op.floordiv,
}

 ## `_eval_math`

 Recursive AST tree walker. Three valid node types:

 - `ast.Constant` — a number literal, return the value directly
 - `ast.BinOp` — binary operation (`a + b`), recurse on both sides, apply operator
 - `ast.UnaryOp` — unary operation (`-x`), same pattern

 Anything else — function calls, imports, attribute access — hits the final
 `raise ValueError` and is rejected before any computation happens.

 **Two safety guards inside `BinOp`:**
 - Division by zero → clean error message instead of `ZeroDivisionError`
 - Exponent > 1000 → `2 ** 10000` is valid Python but produces thousands of digits
   and can freeze the process


In [ ]:
def _eval_math(node: ast.AST) -> Any:
    if isinstance(node, ast.Constant):
        if not isinstance(node.value, (int, float)):
            raise ValueError(f"Type not supported: {type(node.value).__name__}")
        return node.value

    if isinstance(node, ast.BinOp):
        left  = _eval_math(node.left)
        right = _eval_math(node.right)
        if isinstance(node.op, (ast.Div, ast.FloorDiv)) and right == 0:
            raise ValueError("Division by zero")
        if isinstance(node.op, ast.Pow) and abs(right) > 1000:
            raise ValueError(f"Exponent too large: {right}")
        operator_fn = _ALLOWED_OPERATORS.get(type(node.op))
        if operator_fn is None:
            raise ValueError(f"Unsupported operator: {ast.dump(node.op)}")
        return operator_fn(left, right)

    if isinstance(node, ast.UnaryOp):
        operand = _eval_math(node.operand)
        operator_fn = _ALLOWED_OPERATORS.get(type(node.op))
        if operator_fn is None:
            raise ValueError(f"Unsupported unary operator: {ast.dump(node.op)}")
        return operator_fn(operand)

    raise ValueError(f"Unsupported expression: {ast.dump(node)}")

 ## `calculator`

 The `@tool` decorator turns the function into a LangChain tool.
 The docstring becomes the description the LLM reads when deciding whether to call it.

 Errors are returned as strings instead of raising — the agent receives
 `"Calculation error: ..."` as a `ToolMessage` and handles it gracefully
 instead of crashing the graph.

In [ ]:
@tool
async def calculator(expression: str) -> str:
    """
    Safely evaluates a simple mathematical expression.
    Arguments:
        expression: Arithmetic expression to evaluate.
    """
    try:
        parsed = ast.parse(expression, mode="eval")
        result = _eval_math(parsed.body)
        log.info("calculator_completed: expr=%s result=%s", expression, result)
        return str(result)
    except Exception as exc:
        log.error("calculator_error: expr=%s error=%s", expression, str(exc))
        return f"Calculation error: {str(exc)}"

 <div align="center">
   <img src="image/math_tool_1.png" width="800"padding="10"/>
 </div>